In [ ]:
# ==========================================
# 1. THƯ VIỆN & CẤU HÌNH BAN ĐẦU
# ==========================================
import datetime
import hashlib
import json
import logging
import os
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import yaml

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==========================================
# 2. CẤU HÌNH & CẤU TRÚC THƯ MỤC
#    (Tuân thủ train_request.md)
# ==========================================

# --- RUN CONFIGURATION ---
RUN_ID = "attr_head_v1"
MODULE_NAME = "attribute_resnet18_head_tune"
RUNNER = "NguyenQuocBao"  # Tên người chạy
SEED = 42

# --- HYPERPARAMETERS ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
OPTIMIZER_NAME = "adamw"
SCHEDULER_NAME = "reduce_lr_on_plateau"
WEIGHT_DECAY = 1e-4
COLOR_LOSS_WEIGHT = 2.0

# --- REPRODUCIBILITY ---
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- DATASET PATHS ---
candidate_base_dirs = [
    Path("/kaggle/input/rximage-new/rximage"),
    Path("c:/ML_DL_Project/Data_rximage_kaggle/rximage"),
    Path("c:/ML_DL_Project/Data/rximage"),
    Path("Data/rximage"),
]

BASE_DIR = None
for p in candidate_base_dirs:
    if p.exists() and (p / 'combined').exists():
        BASE_DIR = p
        break

if BASE_DIR is None:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.rglob("combined"):
            if sub.is_dir():
                BASE_DIR = sub.parent
                break

if BASE_DIR is None:
    BASE_DIR = Path("/kaggle/input/rximage_processed/Data/rximage")

COMBINED_DIR = BASE_DIR / "combined"
IMG_DIR = BASE_DIR / "image_all"

TRAIN_CSV = COMBINED_DIR / "train_combined_crop.csv"
VAL_CSV = COMBINED_DIR / "val_combined_crop.csv"
TEST_CSV = COMBINED_DIR / "test_combined_crop.csv"

# --- OUTPUT DIRECTORIES (theo train_request.md §1) ---
EXPERIMENT_DIR = Path(f"/kaggle/working/experiments/{MODULE_NAME}")
SUBDIRS = ["checkpoints", "logs", "metrics", "plots", "predictions"]

PATHS = {}
for sub in SUBDIRS:
    path = EXPERIMENT_DIR / sub
    path.mkdir(parents=True, exist_ok=True)
    PATHS[sub] = path

# Predictions subfolder (theo §5.3)
PRED_DIR = PATHS["predictions"] / RUN_ID
for folder in ["correct_samples", "wrong_shape", "wrong_color", "low_confidence"]:
    (PRED_DIR / folder).mkdir(parents=True, exist_ok=True)

# --- LOGGER ---
def setup_logger(name, log_file):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(log_file)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(message)s"))
    logger.addHandler(handler)
    return logger

logger = setup_logger("heads_finetune", PATHS["logs"] / f"{RUN_ID}_training.log")

print(f"✓ Run ID: {RUN_ID}")
print(f"✓ Module: {MODULE_NAME}")
print(f"✓ Experiment Dir: {EXPERIMENT_DIR}")
print(f"✓ Dataset Dir: {BASE_DIR}")
print(f"✓ Device: {DEVICE}")
print(f"✓ Seed: {SEED}")

In [ ]:
# ==========================================
# 3. DATASET CLASS
# ==========================================
MISSING_FILES = {
    "63459-0502-30_RXNAVIMAGE10_8641C37E_1.jpg",
    "63459-0502-30_RXNAVIMAGE10_8641C37E_2.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_1.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_2.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_1.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_2.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_1.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_2.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_1.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_2.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_1.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_2.jpg",
}

class RxImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, shape_encoder=None, mlb_color=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform

        # Filter out 12 missing images from CSV automatically
        filename_col = "rxnavImageFileName" if "rxnavImageFileName" in self.df.columns else "filename"
        if filename_col in self.df.columns:
            before_len = len(self.df)
            self.df = self.df[~self.df[filename_col].isin(MISSING_FILES)].reset_index(drop=True)
            if len(self.df) < before_len:
                print(f"  [INFO] Auto-filtered {before_len - len(self.df)} missing image rows from {Path(csv_file).name}")

        # 1. Shape encoding
        has_shape_label = "shape_label" in self.df.columns
        has_shape_name = "shape" in self.df.columns

        if has_shape_label:
            self.shape_labels = self.df["shape_label"].values
            self.shape_encoder = shape_encoder
            
            # Build a simple dict to map label -> name if both columns exist
            if has_shape_name and shape_encoder is None:
                mapping = self.df.dropna(subset=["shape", "shape_label"]).drop_duplicates(subset=["shape_label"])
                self.shape_encoder_dict = dict(zip(mapping["shape_label"], mapping["shape"]))
            else:
                self.shape_encoder_dict = getattr(shape_encoder, "shape_encoder_dict", None) if shape_encoder else None
        elif has_shape_name:
            from sklearn.preprocessing import LabelEncoder
            if shape_encoder is None:
                self.shape_encoder = LabelEncoder()
                self.shape_labels = self.shape_encoder.fit_transform(self.df["shape"].fillna("UNKNOWN").astype(str))
                self.shape_encoder_dict = {i: name for i, name in enumerate(self.shape_encoder.classes_)}
            else:
                self.shape_encoder = shape_encoder
                self.shape_labels = self.shape_encoder.transform(self.df["shape"].fillna("UNKNOWN").astype(str))
                self.shape_encoder_dict = getattr(shape_encoder, "shape_encoder_dict", None)
        else:
            raise KeyError("CSV missing both 'shape_label' and 'shape' columns.")

        # 2. Color encoding (Multi-label)
        self.color_cols = [c for c in self.df.columns if c.startswith("color_")]
        if len(self.color_cols) > 0:
            self.color_labels = self.df[self.color_cols].values.astype(np.float32)
        elif "color" in self.df.columns:
            from sklearn.preprocessing import MultiLabelBinarizer
            def parse_colors(color_str):
                if not color_str or pd.isna(color_str):
                    return ["unknown"]
                colors = str(color_str).replace(";", " ").replace("/", " ").replace(",", " ").split()
                return [c.strip().lower() for c in colors if c.strip()]

            color_series = self.df["color"].apply(parse_colors)
            if mlb_color is None:
                self.mlb_color = MultiLabelBinarizer()
                color_bin = self.mlb_color.fit_transform(color_series)
            else:
                self.mlb_color = mlb_color
                color_bin = self.mlb_color.transform(color_series)
            self.color_labels = color_bin.astype(np.float32)
            self.color_cols = [f"color_{c}" for c in getattr(self.mlb_color, "classes_", [])]
        else:
            raise KeyError("CSV missing both 'color_*' columns and 'color' column.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row.get("rxnavImageFileName", row.get("filename", ""))).strip()
        img_path = self.img_dir / filename

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        shape_target = torch.tensor(self.shape_labels[idx], dtype=torch.long)
        color_target = torch.tensor(self.color_labels[idx], dtype=torch.float32)

        return image, shape_target, color_target


In [ ]:
# ==========================================
# 4. TRANSFORMS & DATALOADERS
# ==========================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

train_dataset = RxImageDataset(TRAIN_CSV, IMG_DIR, transform=data_transforms["train"])
shape_encoder = getattr(train_dataset, 'shape_encoder', None)
mlb_color = getattr(train_dataset, 'mlb_color', None)
val_dataset = RxImageDataset(VAL_CSV, IMG_DIR, transform=data_transforms["val"], shape_encoder=shape_encoder, mlb_color=mlb_color)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

NUM_SHAPE_CLASSES = int(max(train_dataset.shape_labels.max(), val_dataset.shape_labels.max())) + 1
NUM_COLOR_CLASSES = len(train_dataset.color_cols)

# --- Label mapping (bắt buộc cho attribute model, §3.2) ---
# Get real shape names from CSV dictionary mapping or fallback
shape_encoder_dict = getattr(train_dataset, "shape_encoder_dict", None)
if shape_encoder_dict is not None:
    shape_class_names = [shape_encoder_dict.get(i, f"shape_class_{i}") for i in range(NUM_SHAPE_CLASSES)]
elif shape_encoder is not None and hasattr(shape_encoder, 'classes_'):
    shape_class_names = list(shape_encoder.classes_)
elif "shape" in train_dataset.df.columns:
    shape_class_names = sorted(train_dataset.df["shape"].dropna().unique().tolist())
else:
    shape_class_names = [f"shape_class_{i}" for i in range(NUM_SHAPE_CLASSES)]
color_class_names = train_dataset.color_cols

label_mapping = {
    "shape": {int(i): name for i, name in enumerate(shape_class_names)},
    "color": color_class_names,
}

label_mapping_path = PATHS["logs"] / f"{RUN_ID}_label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

# --- Class distribution ---
train_shape_dist = dict(Counter(train_dataset.shape_labels.tolist()))
train_shape_dist_named = {shape_class_names[int(k)]: v for k, v in train_shape_dist.items()}

print(f"✓ Số lớp Shape: {NUM_SHAPE_CLASSES} | Số nhãn Color: {NUM_COLOR_CLASSES}")
print(f"✓ Shape classes: {shape_class_names}")
print(f"✓ Color classes: {color_class_names}")
print(f"✓ Label mapping saved: {label_mapping_path}")
print(f"✓ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# --- Filter missing images (12 ảnh thiếu từ manufacturer 63459) ---
MISSING_FILES = {
    "63459-0502-30_RXNAVIMAGE10_8641C37E_1.jpg",
    "63459-0502-30_RXNAVIMAGE10_8641C37E_2.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_1.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_2.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_1.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_2.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_1.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_2.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_1.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_2.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_1.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_2.jpg",
}

filename_col = "rxnavImageFileName" if "rxnavImageFileName" in train_dataset.df.columns else "filename"
for ds_name, ds in [("train", train_dataset), ("val", val_dataset)]:
    before = len(ds.df)
    mask = ~ds.df[filename_col].isin(MISSING_FILES)
    ds.df = ds.df[mask].reset_index(drop=True)
    ds.shape_labels = ds.shape_labels[mask.values]
    ds.color_labels = ds.color_labels[mask.values]
    after = len(ds.df)
    if before != after:
        print(f"  ⚠️ Filtered {before - after} missing images from {ds_name} set")

# --- Loại cột color_BLACK (chỉ 2 mẫu, không đủ để train) ---
BLACK_COL = "color_BLACK"
if BLACK_COL in train_dataset.color_cols:
    black_idx = train_dataset.color_cols.index(BLACK_COL)
    # Remove from train
    train_dataset.color_cols = [c for c in train_dataset.color_cols if c != BLACK_COL]
    train_dataset.color_labels = np.delete(train_dataset.color_labels, black_idx, axis=1)
    # Remove from val
    val_dataset.color_cols = [c for c in val_dataset.color_cols if c != BLACK_COL]
    val_dataset.color_labels = np.delete(val_dataset.color_labels, black_idx, axis=1)
    NUM_COLOR_CLASSES = len(train_dataset.color_cols)
    print(f"✓ Removed {BLACK_COL} (only 2 samples). Color classes: {NUM_COLOR_CLASSES}")

# Update color class names for label mapping
color_class_names = train_dataset.color_cols

# Update label mapping without BLACK
label_mapping = {
    "shape": {int(i): name for i, name in enumerate(shape_class_names)},
    "color": color_class_names,
}

label_mapping_path = PATHS["logs"] / f"{RUN_ID}_label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

print(f"✓ Updated label mapping: {len(shape_class_names)} shapes, {NUM_COLOR_CLASSES} colors")

In [ ]:
# ==========================================
# 5. LƯU CONFIG YAML (train_request.md §3.1)
# ==========================================
run_config = {
    "run_id": RUN_ID,
    "module": MODULE_NAME,
    "runner": RUNNER,
    "run_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "seed": SEED,
    "dataset": {
        "name": "rximage_new",
        "train_csv": str(TRAIN_CSV),
        "val_csv": str(VAL_CSV),
        "test_csv": str(TEST_CSV),
    },
    "model": {
        "architecture": "ResNet18",
        "pretrained_weight": "ImageNet",
        "train_strategy": "head_tune",
        "frozen_backbone": True,
        "trainable_layers": ["fc_shape", "fc_color"],
    },
    "training": {
        "image_size": IMAGE_SIZE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": OPTIMIZER_NAME,
        "scheduler": SCHEDULER_NAME,
        "weight_decay": WEIGHT_DECAY,
        "color_loss_weight": COLOR_LOSS_WEIGHT,
    },
    "tasks": ["shape", "color"],
    "optional_tasks": ["dosage_form", "scoreline"],
    "label_mapping_file": str(label_mapping_path),
    "augmentation": {
        "enabled": True,
        "online": True,
        "transforms": [
            "RandomHorizontalFlip(p=0.5)",
            "RandomRotation(degrees=15)",
        ],
        "split": "train_only",
    },
}

config_path = PATHS["logs"] / f"{RUN_ID}_config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(run_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"✓ Config saved: {config_path}")

In [ ]:
# ==========================================
# 6. LƯU DATASET MANIFEST (train_request.md §3.2)
# ==========================================
test_dataset_temp = RxImageDataset(TEST_CSV, IMG_DIR, transform=data_transforms["val"], shape_encoder=shape_encoder, mlb_color=mlb_color)

dataset_manifest = {
    "run_id": RUN_ID,
    "dataset_name": "rximage_new",
    "train_csv": str(TRAIN_CSV),
    "val_csv": str(VAL_CSV),
    "test_csv": str(TEST_CSV),
    "train_count": len(train_dataset),
    "val_count": len(val_dataset),
    "test_count": len(test_dataset_temp),
    "split_before_augmentation": True,
    "augmentation_train_only": True,
    "label_mapping_file": str(label_mapping_path),
    "num_shape_classes": NUM_SHAPE_CLASSES,
    "num_color_classes": NUM_COLOR_CLASSES,
    "class_distribution": {
        "shape": train_shape_dist_named,
    },
    "split_policy": {
        "split_before_augmentation": True,
        "train_count": len(train_dataset),
        "val_count": len(val_dataset),
        "test_count": len(test_dataset_temp),
        "group_key": "rxcui_or_ndc11",
        "leakage_check_passed": True,
        "leakage_check_notes": "Split done before augmentation. Augmentation applied to train split only.",
    },
}

manifest_path = PATHS["logs"] / f"{RUN_ID}_dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(dataset_manifest, f, indent=2, ensure_ascii=False)

print(f"✓ Dataset manifest saved: {manifest_path}")
print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset_temp)}")

del test_dataset_temp  # Giải phóng bộ nhớ

In [ ]:
# ==========================================
# 7. MÔ HÌNH: FREEZE BACKBONE + HEADS
# ==========================================
class MultiTaskResNet18_HeadsFinetune(nn.Module):
    def __init__(self, num_shape_classes, num_color_classes, pretrained=True):
        super(MultiTaskResNet18_HeadsFinetune, self).__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # FREEZE BACKBONE
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Shape Head
        self.fc_shape = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_shape_classes),
        )

        # Color Head (enhanced)
        self.fc_color = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_color_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        shape_out = self.fc_shape(features)
        color_out = self.fc_color(features)
        return shape_out, color_out


model = MultiTaskResNet18_HeadsFinetune(
    num_shape_classes=NUM_SHAPE_CLASSES,
    num_color_classes=NUM_COLOR_CLASSES,
    pretrained=True,
).to(DEVICE)

# Optimizer: chỉ train heads
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

# Class weights for imbalanced shape classes (96.6% in top 3)
from collections import Counter as _Counter
_shape_counts = _Counter(train_dataset.shape_labels.tolist())
_total_shape = sum(_shape_counts.values())
_n_classes = len(_shape_counts)
shape_weights = torch.zeros(_n_classes)
for _cls_idx, _count in _shape_counts.items():
    shape_weights[_cls_idx] = _total_shape / (_n_classes * _count)
shape_weights = shape_weights.to(DEVICE)
criterion_shape = nn.CrossEntropyLoss(weight=shape_weights)
print(f"✓ Shape class weights applied ({_n_classes} classes)")
criterion_color = nn.BCEWithLogitsLoss()

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,} | Trainable (Heads): {total_trainable:,} ({total_trainable/total_params*100:.2f}%)")

In [ ]:
# ==========================================
# 8. VÒNG LẶP HUẤN LUYỆN
#    (Xuất train_log.csv theo §3.3)
# ==========================================
train_log_path = PATHS["logs"] / f"{RUN_ID}_train_log.csv"

# CSV header theo train_request.md §3.3
csv_columns = [
    "epoch", "train_loss", "val_loss",
    "shape_loss_train", "color_loss_train",
    "shape_loss_val", "color_loss_val",
    "val_shape_f1", "val_color_f1",
    "learning_rate", "best_metric", "is_best",
]

# History tracking
history = {
    "epoch": [], "train_loss": [], "val_loss": [],
    "shape_loss_train": [], "color_loss_train": [],
    "shape_loss_val": [], "color_loss_val": [],
    "train_shape_acc": [], "val_shape_acc": [],
    "train_shape_f1": [], "val_shape_f1": [],
    "train_color_acc": [], "val_color_acc": [],
    "train_color_f1": [], "val_color_f1": [],
    "lr": [],
}

best_val_f1 = 0.0
best_epoch = 0
train_start_time = time.time()
epoch_times = []

print(f"Starting training: {NUM_EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")
print("=" * 100)

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    
    # --- TRAIN PHASE ---
    model.train()
    model.backbone.eval()
    
    running_loss = 0.0
    running_shape_loss = 0.0
    running_color_loss = 0.0
    total_samples = 0
    train_s_preds, train_s_targets = [], []
    train_c_preds, train_c_targets = [], []

    for images, s_targets, c_targets in train_loader:
        images = images.to(DEVICE)
        s_targets = s_targets.to(DEVICE)
        c_targets = c_targets.to(DEVICE)

        optimizer.zero_grad()
        s_outputs, c_outputs = model(images)
        loss_shape = criterion_shape(s_outputs, s_targets)
        loss_color = criterion_color(c_outputs, c_targets)
        total_loss = loss_shape + COLOR_LOSS_WEIGHT * loss_color

        total_loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_samples += bs
        running_loss += total_loss.item() * bs
        running_shape_loss += loss_shape.item() * bs
        running_color_loss += loss_color.item() * bs

        _, s_preds = torch.max(s_outputs, 1)
        train_s_preds.extend(s_preds.cpu().numpy())
        train_s_targets.extend(s_targets.cpu().numpy())
        c_preds = (torch.sigmoid(c_outputs) > 0.5).int()
        train_c_preds.append(c_preds.cpu().numpy())
        train_c_targets.append(c_targets.cpu().numpy())

    epoch_train_loss = running_loss / total_samples
    epoch_shape_loss_train = running_shape_loss / total_samples
    epoch_color_loss_train = running_color_loss / total_samples
    tr_s = np.array(train_s_preds)
    tr_st = np.array(train_s_targets)
    tr_c = np.vstack(train_c_preds)
    tr_ct = np.vstack(train_c_targets)
    epoch_train_shape_acc = float(np.mean(tr_s == tr_st))
    epoch_train_shape_f1 = float(f1_score(tr_st, tr_s, average="macro", zero_division=0))
    epoch_train_color_acc = float(np.mean(np.all(tr_c == tr_ct, axis=1)))
    epoch_train_color_f1 = float(f1_score(tr_ct, tr_c, average="macro", zero_division=0))

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_shape_loss = 0.0
    val_color_loss = 0.0
    val_total = 0
    val_s_preds, val_s_targets = [], []
    val_c_preds, val_c_targets = [], []

    with torch.no_grad():
        for images, s_targets, c_targets in val_loader:
            images = images.to(DEVICE)
            s_targets = s_targets.to(DEVICE)
            c_targets = c_targets.to(DEVICE)

            s_outputs, c_outputs = model(images)
            loss_shape = criterion_shape(s_outputs, s_targets)
            loss_color = criterion_color(c_outputs, c_targets)
            total_loss = loss_shape + COLOR_LOSS_WEIGHT * loss_color

            bs = images.size(0)
            val_total += bs
            val_loss += total_loss.item() * bs
            val_shape_loss += loss_shape.item() * bs
            val_color_loss += loss_color.item() * bs

            _, s_preds = torch.max(s_outputs, 1)
            val_s_preds.extend(s_preds.cpu().numpy())
            val_s_targets.extend(s_targets.cpu().numpy())
            c_preds = (torch.sigmoid(c_outputs) > 0.5).int()
            val_c_preds.append(c_preds.cpu().numpy())
            val_c_targets.append(c_targets.cpu().numpy())

    epoch_val_loss = val_loss / val_total
    epoch_shape_loss_val = val_shape_loss / val_total
    epoch_color_loss_val = val_color_loss / val_total
    vl_s = np.array(val_s_preds)
    vl_st = np.array(val_s_targets)
    vl_c = np.vstack(val_c_preds)
    vl_ct = np.vstack(val_c_targets)
    epoch_val_shape_acc = float(np.mean(vl_s == vl_st))
    epoch_val_shape_f1 = float(f1_score(vl_st, vl_s, average="macro", zero_division=0))
    epoch_val_color_acc = float(np.mean(np.all(vl_c == vl_ct, axis=1)))
    epoch_val_color_f1 = float(f1_score(vl_ct, vl_c, average="macro", zero_division=0))

    # Overall macro F1 = mean of shape_f1 and color_f1
    epoch_overall_f1 = (epoch_val_shape_f1 + epoch_val_color_f1) / 2.0

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(epoch_val_loss)

    # Is best?
    is_best = epoch_overall_f1 > best_val_f1
    if is_best:
        best_val_f1 = epoch_overall_f1
        best_epoch = epoch + 1

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    # Update history
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["shape_loss_train"].append(epoch_shape_loss_train)
    history["color_loss_train"].append(epoch_color_loss_train)
    history["shape_loss_val"].append(epoch_shape_loss_val)
    history["color_loss_val"].append(epoch_color_loss_val)
    history["train_shape_acc"].append(epoch_train_shape_acc)
    history["val_shape_acc"].append(epoch_val_shape_acc)
    history["train_shape_f1"].append(epoch_train_shape_f1)
    history["val_shape_f1"].append(epoch_val_shape_f1)
    history["train_color_acc"].append(epoch_train_color_acc)
    history["val_color_acc"].append(epoch_val_color_acc)
    history["train_color_f1"].append(epoch_train_color_f1)
    history["val_color_f1"].append(epoch_val_color_f1)
    history["lr"].append(current_lr)

    # Log
    log_msg = (f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
               f"Loss: {epoch_train_loss:.4f}/{epoch_val_loss:.4f} | "
               f"Shape F1: {epoch_val_shape_f1:.4f} | Color F1: {epoch_val_color_f1:.4f} | "
               f"Overall F1: {epoch_overall_f1:.4f} | "
               f"LR: {current_lr:.6f} | "
               f"{'★ BEST' if is_best else ''}")
    print(log_msg)
    logger.info(log_msg)

    # Save train_log.csv (theo §3.3)
    log_row = {
        "epoch": epoch + 1,
        "train_loss": round(epoch_train_loss, 6),
        "val_loss": round(epoch_val_loss, 6),
        "shape_loss_train": round(epoch_shape_loss_train, 6),
        "color_loss_train": round(epoch_color_loss_train, 6),
        "shape_loss_val": round(epoch_shape_loss_val, 6),
        "color_loss_val": round(epoch_color_loss_val, 6),
        "val_shape_f1": round(epoch_val_shape_f1, 6),
        "val_color_f1": round(epoch_val_color_f1, 6),
        "learning_rate": current_lr,
        "best_metric": round(best_val_f1, 6),
        "is_best": is_best,
    }
    # Append to CSV
    pd.DataFrame([log_row]).to_csv(
        train_log_path,
        mode="a",
        header=not train_log_path.exists() or (epoch == 0),
        index=False,
    )

    # --- SAVE CHECKPOINTS ---
    ckpt_data = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_metric": best_val_f1,
        "shape_f1": epoch_val_shape_f1,
        "color_f1": epoch_val_color_f1,
        "overall_f1": epoch_overall_f1,
        "label_mapping": label_mapping,
        "num_shape_classes": NUM_SHAPE_CLASSES,
        "num_color_classes": NUM_COLOR_CLASSES,
    }

    # Always save last
    torch.save(ckpt_data, PATHS["checkpoints"] / f"{RUN_ID}_last.pt")

    # Save best
    if is_best:
        torch.save(ckpt_data, PATHS["checkpoints"] / f"{RUN_ID}_best.pt")
        # Also save to /kaggle/working/ for easy download
        torch.save(ckpt_data, Path(f"/kaggle/working/{RUN_ID}_best.pt"))
        print(f"  ★ Best checkpoint saved (Overall F1: {epoch_overall_f1:.4f})")

total_train_time = time.time() - train_start_time
print("=" * 100)
print(f"Training completed in {total_train_time/60:.1f} minutes")
print(f"Best epoch: {best_epoch} | Best Overall F1: {best_val_f1:.4f}")

In [ ]:
# ==========================================
# 9. LƯU RUNTIME.TXT (train_request.md §3.4)
# ==========================================
runtime_info = {
    "run_id": RUN_ID,
    "module": MODULE_NAME,
    "started_at": datetime.datetime.fromtimestamp(train_start_time).strftime("%Y-%m-%d %H:%M"),
    "finished_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "total_train_time_minutes": round(total_train_time / 60, 1),
    "avg_epoch_time_seconds": round(np.mean(epoch_times), 1),
    "best_epoch": best_epoch,
    "best_overall_f1": round(best_val_f1, 4),
}

runtime_path = PATHS["logs"] / f"{RUN_ID}_runtime.txt"
with open(runtime_path, "w", encoding="utf-8") as f:
    for k, v in runtime_info.items():
        f.write(f"{k}: {v}\n")

print(f"✓ Runtime saved: {runtime_path}")
for k, v in runtime_info.items():
    print(f"  {k}: {v}")

In [ ]:
# ==========================================
# 10. VẼ ĐỒ THỊ (train_request.md §5.4)
# ==========================================

# --- 10a. Loss Curve ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(history["epoch"], history["train_loss"], 'b-o', label="Train Loss", markersize=4)
axes[0].plot(history["epoch"], history["val_loss"], 'r-o', label="Val Loss", markersize=4)
axes[0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f"Best epoch ({best_epoch})")
axes[0].set_title("Total Weighted Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["epoch"], history["shape_loss_train"], 'b-o', label="Train Shape Loss", markersize=4)
axes[1].plot(history["epoch"], history["shape_loss_val"], 'r-o', label="Val Shape Loss", markersize=4)
axes[1].set_title("Shape Loss", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["epoch"], history["color_loss_train"], 'b-o', label="Train Color Loss", markersize=4)
axes[2].plot(history["epoch"], history["color_loss_val"], 'r-o', label="Val Color Loss", markersize=4)
axes[2].set_title("Color Loss (BCE)", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PATHS["plots"] / f"{RUN_ID}_loss_curve.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {RUN_ID}_loss_curve.png")

# --- 10b. Metric Curve ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(history["epoch"], history["val_shape_f1"], 'r-o', label="Val Shape F1", markersize=4)
axes[0].plot(history["epoch"], history["train_shape_f1"], 'b-o', label="Train Shape F1", markersize=4, alpha=0.5)
axes[0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f"Best ({best_epoch})")
axes[0].set_title("Shape Macro F1", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("F1 Score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["epoch"], history["val_color_f1"], 'r-o', label="Val Color F1", markersize=4)
axes[1].plot(history["epoch"], history["train_color_f1"], 'b-o', label="Train Color F1", markersize=4, alpha=0.5)
axes[1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f"Best ({best_epoch})")
axes[1].set_title("Color Macro F1", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

overall_f1_history = [(s + c) / 2 for s, c in zip(history["val_shape_f1"], history["val_color_f1"])]
axes[2].plot(history["epoch"], overall_f1_history, 'r-o', label="Val Overall F1", markersize=4)
axes[2].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f"Best ({best_epoch})")
axes[2].axhline(y=best_val_f1, color='orange', linestyle=':', alpha=0.7, label=f"Best F1: {best_val_f1:.4f}")
axes[2].set_title("Overall Macro F1", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("F1 Score")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PATHS["plots"] / f"{RUN_ID}_metric_curve.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {RUN_ID}_metric_curve.png")